In [ ]:
# subjects : 100003.nii.gz  100006.nii.gz  100009.nii.gz  100020.nii.gz  100027.nii.gz  100005.nii.gz  100008.nii.gz  100015.nii.gz  100024.nii.gz  100029.nii.gz

In [ ]:
ORIG_DIR = "/nfs/data/nii/data1/camaret___whole_body_benchmark"
ROOT = "/nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/patchwork_inference_benchmark"

# copy 10 subjects from ORIG_DIR to IMG_DIR, renaming to 100000.nii.gz, 100001.nii.gz, ...
import shutil
from pathlib import Path
subjects = ["100003", "100006", "100009", "100020", "100027", "100005", "100008", "100015", "100024", "100029"]
for i, subj in enumerate(subjects):
    #img
    src = Path(ORIG_DIR) / f"{subj}/NII/{subj}_img.nii"
    dst = Path(ROOT) / f"imagesTs/{subj}.nii"
    shutil.copy(src, dst)
    #mask
    src = Path(ORIG_DIR) / f"{subj}/NII/{subj}_mask.nii.gz"
    dst = Path(ROOT) / f"labelsTs/{subj}.nii.gz"
    shutil.copy(src, dst)

In [10]:
# open 1rst mask and check that it has 14 classes (0-13)
import nibabel as nib
import numpy as np
mask_path = Path(ROOT) / "labelsTs/100003.nii.gz"
mask_img = nib.load(mask_path)
mask_data = mask_img.get_fdata()
print("Unique labels in mask:", np.unique(mask_data))
pred_path = Path(ROOT) / "predsTs/100003.nii"
pred_img = nib.load(pred_path)
pred_data = pred_img.get_fdata()
print("Unique labels in pred:", np.unique(pred_data))
print("Number of unique labels in pred:", len(np.unique(pred_data)))

Unique labels in mask: [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 13. 14. 15. 16. 17. 18.
 19. 20. 21. 22. 23. 24. 25.]
Unique labels in pred: [0.000e+00 1.000e+00 2.000e+00 ... 9.997e+03 9.998e+03 9.999e+03]
Number of unique labels in pred: 7620


In [9]:
len(np.unique(pred_data))

7620

python scripts/nnunet_patchwork_comp/eval.py \
    --gt-pattern   /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/patchwork_inference_benchmark/labelsTs/{subject}.nii.gz \
    --pred-pattern /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/patchwork_inference_benchmark/predsTs/{subject}.nii.gz \
    --subjects     /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/patchwork_inference_benchmark/predsTs \
    --output       results/inference_benchmark/patchwork.json --no-nsd

---
  Parameters That Impact Prediction Time

  The Inference Loop Structure

  With generate_type='random', the code restructures the loop at model.py:832-835:

  reps = repetitions  →  reps = 4
  repetitions = 1

  outer loop: num_chunks (20 iterations)
    inner loop: repetitions (1 iteration)
      → sample reps (4) patches at level 0
      → expand per level using branch_factor
      → forward pass through network
      → stitch results

  So the network is called 20 times (once per chunk). Each call processes a full patch tree.

  ---
  Patch Count Formula
  
  At each depth level, patch count multiplies:

  ┌───────┬───────────────────────────────┬────────────────────┐
  │ Level │       Patches per chunk       │ Total (×20 chunks) │
  ├───────┼───────────────────────────────┼────────────────────┤
  │ 0     │ reps = 4                      │ 80                 │
  ├───────┼───────────────────────────────┼────────────────────┤
  │ 1     │ 4 × branch_factor[0]=4 → 16   │ 320                │
  ├───────┼───────────────────────────────┼────────────────────┤
  │ 2     │ 16 × branch_factor[1]=4 → 64  │ 1,280              │
  ├───────┼───────────────────────────────┼────────────────────┤
  │ 3     │ 64 × branch_factor[2]=8 → 512 │ 10,240             │
  └───────┴───────────────────────────────┴────────────────────┘

  The deepest level dominates computation. Total fine-level patches = num_chunks × repetitions × ∏(branch_factor).

  ---
  Parameters Ranked by Time Impact

  1. branch_factor — Exponential impact

  The most powerful lever. Each element multiplies the patch count at that level. Your [4,4,8] gives 128× more patches at the fine level than at the coarse level. Halving the last
  factor ([4,4,4]) cuts fine-level patches by 2×. Reducing depth by removing the last factor entirely cuts patches by 8×.

  2. num_chunks — Linear, multiplies everything

  Directly multiplies the total number of network calls. num_chunks=20 means 20 forward passes. Cutting to 10 halves total time. Note: this parameter exists to control GPU memory (each
  chunk processes one batch), not sampling quality — splitting the same patches into more chunks does not add accuracy.

  3. repetitions — Linear, only at coarse level

  Sets the number of level-0 patches per chunk (reps=4). Propagates linearly through all levels via multiplication with branch_factor. Halving it halves the total patch count at every
  level.

  4. level='mix' — Additive overhead for stitching

  Requires stitching and upsampling predictions from all depth levels, then combining them (model.py:1017-1040). A single-level output (e.g., level=-1) would only stitch the deepest
  level. The mixing step includes a resizeNDlinear call per level, which has non-trivial cost for large 3D volumes.

  5. scale_to_original=False — Minor, saves one resize

  Already set to False, avoiding a final 3D resize of the output. Setting it to True would add one resizeNDlinear call.

  6. lazyEval (commented out) — Could be a major speedup

  If enabled, it computes attention scores after each level and only forwards the top fraction of patches to the next level. lazyEval=0.7 would keep only 70% of patches at each level
  transition. This would cut fine-level computation significantly at the cost of potentially missing low-attention regions.

  7. augment={} — No impact here

  Empty dict means no test-time augmentation. If populated, it would add augmented forward passes.

  8. QMapply_paras — Negligible

  The QMembedding.apply() call is a matrix multiply over the embedding space, run once on CPU after all patches are stitched. Not a bottleneck.

  9. out_typ='atls' / generate_type — Negligible

  Post-processing (argmax, thresholding) is cheap. generate_type='random' patch sampling itself is fast compared to network inference.

  ---
  Accuracy Impact (for the study)

  ┌─────────────────────────────┬────────────────┬───────────────────────────────────────────────────────┐
  │          Parameter          │      Time      │                    Accuracy effect                    │
  ├─────────────────────────────┼────────────────┼───────────────────────────────────────────────────────┤
  │ repetitions ↑               │ linear ↑       │ Better stochastic coverage, less variance             │
  ├─────────────────────────────┼────────────────┼───────────────────────────────────────────────────────┤
  │ branch_factor[-1] ↑         │ linear ↑       │ Finer-level density, detail recovery                  │
  ├─────────────────────────────┼────────────────┼───────────────────────────────────────────────────────┤
  │ num_chunks ↑                │ linear ↑       │ Acts like more repetitions (independent random draws) │
  ├─────────────────────────────┼────────────────┼───────────────────────────────────────────────────────┤
  │ level='mix' vs single level │ adds stitching │ Typically better — uses coarse-to-fine context        │
  ├─────────────────────────────┼────────────────┼───────────────────────────────────────────────────────┤
  │ lazyEval ↓                  │ major ↓        │ Risk of missed regions if fraction too low            │
  ├─────────────────────────────┼────────────────┼───────────────────────────────────────────────────────┤
  │ branch_factor[0] ↑          │ linear ↑       │ More coarse context, diminishing returns quickly      │
  └─────────────────────────────┴────────────────┴───────────────────────────────────────────────────────┘

  ---
  Suggested Study Axes

  The cleanest experiment design would be to vary these three independent axes:

  1. Total patch budget at the deepest level: num_chunks × repetitions × ∏(branch_factor) — sweep from 512 to 20,480
  2. Distribution of that budget: e.g. more chunks vs. more branch depth vs. more repetitions, at fixed total
  3. lazyEval fraction (0.3–0.9) at fixed total budget, measuring the accuracy/speed tradeoff of attention-guided skipping

  This isolates whether it's the total number of patches or how they are structured that drives accuracy.